In [1]:
# ============================================================
#  CodeT5-Small Fine-Tuning trên Spider (Text-to-SQL)
#  Tối ưu cho: Ít Epoch – Độ chính xác cao nhất
#  Model: Salesforce/codet5-small (~60M params, pretrain trên code)
#  CHẠY TRÊN GOOGLE COLAB - ĐÃ TẮT EARLY STOPPING
# ============================================================

!pip install -q "transformers>=4.41.0,<5.0.0" datasets sentencepiece accelerate tqdm
!pip install -q kaggle

import os, json, re, shutil, random, math, time, zipfile, urllib.request
import numpy as np
import nltk
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.amp import GradScaler, autocast
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
    DataCollatorForSeq2Seq,
)
from tqdm.auto import tqdm

# ── 1. SETUP ──────────────────────────────────────────────────
os.environ['KAGGLE_USERNAME'] = "phankhaclap"
os.environ['KAGGLE_KEY']      = "0ba946628cb1f5acb76ecd357f590e95"

# Chuyển sang đường dẫn của Google Colab
FINAL_SAVE_PATH = "/content/CodeT5-small"
CHECKPOINT_DIR  = os.path.join(FINAL_SAVE_PATH, "checkpoints")
RESUME_DIR      = os.path.join(FINAL_SAVE_PATH, "resume")

for d in [FINAL_SAVE_PATH, CHECKPOINT_DIR, RESUME_DIR]:
    os.makedirs(d, exist_ok=True)

RESUME_STATE_FILE = os.path.join(RESUME_DIR, "training_state.json")
RESUME_MODEL_DIR  = os.path.join(RESUME_DIR, "model")
RESUME_OPT_FILE   = os.path.join(RESUME_DIR, "optimizer.pt")

# ── 2. TẢI DỮ LIỆU ────────────────────────────────────────────
print(">>> [1/7] Tải dữ liệu Spider...")
if os.path.exists('spider_data'):
    shutil.rmtree('spider_data')

!kaggle datasets download -d jeromeblanchet/yale-universitys-spider-10-nlp-dataset
zip_path = "yale-universitys-spider-10-nlp-dataset.zip"

if os.path.exists(zip_path):
    print("Đang giải nén dữ liệu...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall("temp_spider")
    if os.path.exists("temp_spider/spider"):
        shutil.move("temp_spider/spider", "spider_data")
    else:
        shutil.rename("temp_spider", "spider_data")
    if os.path.exists('temp_spider'):
        shutil.rmtree('temp_spider', ignore_errors=True)
    os.remove(zip_path)
else:
    print("❌ KHÔNG TÌM THẤY FILE ZIP!")

print("Đang tải công cụ chấm điểm chính thức của Spider...")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/taoyds/spider/master/evaluation.py",
    "evaluation.py"
)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/taoyds/spider/master/process_sql.py",
    "process_sql.py"
)

nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print("Xong bước tải dữ liệu.")

# ── 3. CẤU HÌNH ───────────────────────────────────────────────
CFG = dict(
    model_name      = "Salesforce/codet5-small",
    max_input_len   = 512,
    max_target_len  = 256,
    batch_size      = 8,
    grad_accum      = 4,
    num_epochs      = 5, # Sẽ chạy đúng 30 epoch
    lr              = 2e-4,
    warmup_ratio    = 0.06,
    weight_decay    = 0.01,
    fp16            = True,
    beam_size       = 6,
    patience        = 3,  # Cấu hình còn lại nhưng Early Stopping đã bị tắt ở vòng lặp
    seed            = 42,
    label_smoothing = 0.1,
    question_budget = 96,
    length_penalty  = 0.8,
    num_workers     = 2,  # Tăng lên 2 trên colab để tăng tốc dataloader
)

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])

DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_FP16 = CFG['fp16'] and (DEVICE.type == 'cuda')
print(f"Thiết bị: {DEVICE}  |  Dùng FP16: {USE_FP16}")

# ── 4. TIỀN XỬ LÝ ────────────────────────────────────────────
print(">>> [2/7] Tiền xử lý dữ liệu...")

def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

TYPE_SHORT = {"text": "T", "number": "N", "time": "D", "boolean": "B", "others": "O"}

def build_schema_map(tables_path):
    tables = load_json(tables_path)
    schema_map = {}
    for db in tables:
        db_id     = db['db_id']
        col_types = db.get('column_types', [])
        parts     = []
        for t_idx, t_name in enumerate(db['table_names_original']):
            cols = []
            for c_idx, (t_i, c_name) in enumerate(db['column_names_original']):
                if t_i == t_idx:
                    ct = TYPE_SHORT.get(
                        col_types[c_idx] if c_idx < len(col_types) else "others", "O"
                    )
                    cols.append(f"{c_name}:{ct}")
            parts.append(f"{t_name}({','.join(cols)})")
        schema_map[db_id] = " | ".join(parts)
    return schema_map

def normalize_sql(sql: str) -> str:
    sql = sql.lower().strip()
    sql = re.sub(r'\s+', ' ', sql)
    sql = re.sub(r'\s*([,\(\)])\s*', r' \1 ', sql)
    return re.sub(r'\s+', ' ', sql).strip()

PREFIX     = "Translate English to SQL: "
SEP        = " | schema: "
PREFIX_TOK = 8

def build_input_smart(question: str, schema: str, tokenizer=None, max_len=512) -> str:
    if tokenizer is None:
        return f"{PREFIX}{question.strip()}{SEP}{schema}"
    q_budget = CFG['question_budget']
    s_budget = max_len - PREFIX_TOK - q_budget
    q_ids = tokenizer.encode(
        question.strip(), add_special_tokens=False,
        max_length=q_budget, truncation=True
    )
    s_ids = tokenizer.encode(
        schema, add_special_tokens=False,
        max_length=s_budget, truncation=True
    )
    q_text = tokenizer.decode(q_ids, skip_special_tokens=True)
    s_text = tokenizer.decode(s_ids, skip_special_tokens=True)
    return f"{PREFIX}{q_text}{SEP}{s_text}"

def load_spider_split(data_path, schema_map, tokenizer=None):
    samples = []
    for item in load_json(data_path):
        db_id  = item['db_id']
        schema = schema_map.get(db_id, "")
        inp    = build_input_smart(item['question'], schema, tokenizer, CFG['max_input_len'])
        samples.append({
            "input":    inp,
            "target":   item['query'],
            "db_id":    db_id,
            "question": item['question']
        })
    return samples

# Đường dẫn Colab cho Spider dataset
schema_map = build_schema_map("spider_data/tables.json")
tokenizer  = AutoTokenizer.from_pretrained(CFG['model_name'])

train_data = load_spider_split("spider_data/train_spider.json", schema_map, tokenizer)
dev_data   = load_spider_split("spider_data/dev.json", schema_map, tokenizer)

others_path = "spider_data/train_others.json"
if os.path.exists(others_path):
    train_data += load_spider_split(others_path, schema_map, tokenizer)

random.shuffle(train_data)
print(f"Train: {len(train_data)} mẫu  |  Dev: {len(dev_data)} mẫu")

# ── 5. DATASET & DATALOADER ───────────────────────────────────
print(">>> [3/7] Đóng gói Dữ liệu (Tokenization)...")

class SpiderDataset(Dataset):
    def __init__(self, samples, tokenizer, max_in, max_out):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.max_in    = max_in
        self.max_out   = max_out

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        enc = self.tokenizer(
            s['input'], max_length=self.max_in, truncation=True, padding=False
        )
        tgt = self.tokenizer(
            text_target=s['target'], max_length=self.max_out, truncation=True, padding=False
        )
        labels = [
            l if l != self.tokenizer.pad_token_id else -100
            for l in tgt['input_ids']
        ]
        return {
            'input_ids':      enc['input_ids'],
            'attention_mask': enc['attention_mask'],
            'labels':         labels,
            'target_sql':     s['target'],
            'db_id':          s['db_id'],
            'question':       s['question']
        }

train_ds = SpiderDataset(train_data, tokenizer, CFG['max_input_len'], CFG['max_target_len'])
dev_ds   = SpiderDataset(dev_data,   tokenizer, CFG['max_input_len'], CFG['max_target_len'])

collator = DataCollatorForSeq2Seq(
    tokenizer, model=None, label_pad_token_id=-100, pad_to_multiple_of=8
)

def collate_fn(batch):
    meta     = {k: [b.pop(k) for b in batch] for k in ('target_sql', 'db_id', 'question')}
    collated = collator(batch)
    collated.update(meta)
    return collated

train_loader = DataLoader(
    train_ds,
    batch_size=CFG['batch_size'],
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=CFG['num_workers'],
    pin_memory=True
)
dev_loader = DataLoader(
    dev_ds,
    batch_size=CFG['batch_size'] * 2,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=CFG['num_workers'],
    pin_memory=True
)

# ── 6. LOSS FUNCTION ──────────────────────────────────────────
loss_fn = nn.CrossEntropyLoss(
    label_smoothing=CFG['label_smoothing'], ignore_index=-100
)

def compute_loss(logits, labels):
    B, T, V = logits.shape
    return loss_fn(logits.reshape(B * T, V), labels.reshape(B * T))

# ── 7. MODEL & OPTIMIZER ──────────────────────────────────────
print(f">>> [4/7] Khởi tạo CodeT5Small (60M params, pretrain code)...")

total_steps  = math.ceil(len(train_loader) / CFG['grad_accum']) * CFG['num_epochs']
warmup_steps = int(total_steps * CFG['warmup_ratio'])
no_decay     = ["bias", "LayerNorm.weight"]

def build_optimizer_scheduler(model):
    param_groups = [
        {
            "params": [
                p for n, p in model.named_parameters()
                if not any(nd in n for nd in no_decay)
            ],
            "weight_decay": CFG['weight_decay']
        },
        {
            "params": [
                p for n, p in model.named_parameters()
                if any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0
        },
    ]
    opt   = AdamW(param_groups, lr=CFG['lr'], eps=1e-8, betas=(0.9, 0.98))
    sched = get_cosine_schedule_with_warmup(opt, warmup_steps, total_steps)
    return opt, sched

START_EPOCH, best_em, best_epoch, patience_cnt, history, global_step = 1, 0.0, 0, 0, [], 0

if os.path.exists(RESUME_STATE_FILE) and os.path.exists(RESUME_MODEL_DIR):
    print("🔄 Resume checkpoint phát hiện...")
    state                = load_json(RESUME_STATE_FILE)
    model                = AutoModelForSeq2SeqLM.from_pretrained(RESUME_MODEL_DIR).to(DEVICE)
    optimizer, scheduler = build_optimizer_scheduler(model)
    if os.path.exists(RESUME_OPT_FILE):
        opt_ckpt = torch.load(RESUME_OPT_FILE, map_location=DEVICE)
        optimizer.load_state_dict(opt_ckpt['optimizer'])
        scheduler.load_state_dict(opt_ckpt['scheduler'])
    scaler = GradScaler('cuda', enabled=USE_FP16)
    if 'opt_ckpt' in locals() and 'scaler' in opt_ckpt:
        scaler.load_state_dict(opt_ckpt['scaler'])
    global_step  = state['global_step']
    START_EPOCH  = state['epoch'] + 1
    best_em      = state['best_em']
    best_epoch   = state['best_epoch']
    patience_cnt = state['patience_cnt']
    history      = state.get('history', [])
else:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        CFG['model_name'],
        use_safetensors=True
    ).to(DEVICE)
    optimizer, scheduler = build_optimizer_scheduler(model)
    scaler = GradScaler('cuda', enabled=USE_FP16)

def save_resume_checkpoint(model, optimizer, scheduler, scaler, epoch,
                           global_step, best_em, best_epoch, patience_cnt, history):
    model.save_pretrained(RESUME_MODEL_DIR)
    tokenizer.save_pretrained(RESUME_MODEL_DIR)
    torch.save({
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler':    scaler.state_dict()
    }, RESUME_OPT_FILE)
    with open(RESUME_STATE_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'epoch':        epoch,
            'global_step':  global_step,
            'best_em':      best_em,
            'best_epoch':   best_epoch,
            'patience_cnt': patience_cnt,
            'history':      history,
            'config':       CFG
        }, f, indent=2)

# ── 8. EVAL ───────────────────────────────────────────────────
def evaluate_em(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch in loader:
            generated = model.generate(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                max_new_tokens=CFG['max_target_len'],
                num_beams=CFG['beam_size'],
                early_stopping=True,
                length_penalty=CFG['length_penalty'],
            )
            preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
            for pred, gold in zip(preds, batch['target_sql']):
                correct += int(normalize_sql(pred) == normalize_sql(gold))
                total   += 1
    model.train()
    return correct / total if total > 0 else 0.0

# ── 8b. PERFORMANCE METRICS ───────────────────────────────────
def measure_model_size_mb(model_path: str) -> float:
    total_bytes = 0
    for fname in os.listdir(model_path):
        if fname.endswith(('.bin', '.safetensors', '.pt')):
            total_bytes += os.path.getsize(os.path.join(model_path, fname))
    return total_bytes / (1024 ** 2)


def measure_latency_throughput(model, loader, n_warmup_batches: int = 3):
    model.eval()
    latencies   = []
    total_samps = 0

    with torch.no_grad():
        for i, batch in enumerate(tqdm(loader, desc="⏱ Đo Latency/Throughput", leave=False)):
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            bsz            = input_ids.size(0)

            if i < n_warmup_batches:
                model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=CFG['max_target_len'],
                    num_beams=CFG['beam_size'],
                    early_stopping=True,
                    length_penalty=CFG['length_penalty'],
                )
                continue

            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            t_start = time.perf_counter()

            model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=CFG['max_target_len'],
                num_beams=CFG['beam_size'],
                early_stopping=True,
                length_penalty=CFG['length_penalty'],
            )

            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            t_end = time.perf_counter()

            latencies.append(t_end - t_start)
            total_samps += bsz

    model.train()
    if not latencies:
        return 0.0, 0.0

    total_time_s      = sum(latencies)
    latency_ms_sample = (total_time_s / total_samps) * 1000
    throughput_sps    = total_samps / total_time_s
    return round(latency_ms_sample, 2), round(throughput_sps, 2)


def measure_peak_vram_mb(model, loader, n_batches: int = 5) -> float:
    if DEVICE.type != 'cuda':
        return 0.0

    model.eval()
    torch.cuda.reset_peak_memory_stats(DEVICE)

    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches:
                break
            model.generate(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                max_new_tokens=CFG['max_target_len'],
                num_beams=CFG['beam_size'],
                early_stopping=True,
                length_penalty=CFG['length_penalty'],
            )

    peak_bytes = torch.cuda.max_memory_allocated(DEVICE)
    model.train()
    return round(peak_bytes / (1024 ** 2), 2)


def print_performance_report(metrics: dict):
    print("\n" + "=" * 55)
    print("📊 PERFORMANCE METRICS REPORT")
    print("=" * 55)
    print(f"{'Chỉ số':<28} {'Giá trị':>15}")
    print("-" * 55)
    print(f"{'Model Size (MB)':<28} {metrics['model_size_mb']:>14.2f} MB")
    print(f"{'Latency (ms/sample)':<28} {metrics['latency_ms']:>14.2f} ms")
    print(f"{'Throughput (samples/s)':<28} {metrics['throughput_sps']:>14.2f} s/s")
    vram_str = (
        f"{metrics['peak_vram_mb']:.2f} MB"
        if metrics['peak_vram_mb'] > 0 else "N/A (CPU)"
    )
    print(f"{'Peak VRAM (MB)':<28} {vram_str:>15}")
    print("=" * 55 + "\n")

# ── 9. TRAINING LOOP ──────────────────────────────────────────
print(f"\n>>> [5/7] Bắt đầu huấn luyện ({CFG['num_epochs']} epochs, LR={CFG['lr']:.0e})...")
print(f"Tổng optimizer steps: {total_steps} | Warmup: {warmup_steps} steps")

for epoch in range(START_EPOCH, CFG['num_epochs'] + 1):
    model.train()
    epoch_loss, t0 = 0.0, time.time()
    optimizer.zero_grad()
    train_pbar = tqdm(
        train_loader, desc=f"Epoch {epoch:02d}/{CFG['num_epochs']}", leave=False
    )

    for step, batch in enumerate(train_pbar):
        with autocast('cuda', enabled=USE_FP16):
            outputs = model(
                input_ids=batch['input_ids'].to(DEVICE),
                attention_mask=batch['attention_mask'].to(DEVICE),
                labels=batch['labels'].to(DEVICE),
            )
            loss = compute_loss(
                outputs.logits, batch['labels'].to(DEVICE)
            ) / CFG['grad_accum']

        scaler.scale(loss).backward()

        if (step + 1) % CFG['grad_accum'] == 0 or (step + 1) == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1

        epoch_loss += loss.item() * CFG['grad_accum']
        train_pbar.set_postfix({'loss': f"{(epoch_loss / (step + 1)):.4f}"})

    avg_loss = epoch_loss / len(train_loader)
    print("Đang chạy đánh giá (Evaluation) trên Dev set...")
    em      = evaluate_em(model, dev_loader)
    cur_lr  = scheduler.get_last_lr()[0]
    elapsed = time.time() - t0

    history.append({
        "epoch": epoch,
        "loss":  round(avg_loss, 4),
        "em":    round(em, 4),
        "lr":    round(cur_lr, 8)
    })
    print(
        f"📊 Epoch {epoch:3d}/{CFG['num_epochs']}: "
        f"Loss={avg_loss:.4f} | EM={em:.4f} | LR={cur_lr:.2e} | {elapsed:.0f}s"
    )

    if math.isnan(avg_loss):
        print("❌ Loss = NaN, dừng huấn luyện.")
        break

    if em > best_em:
        best_em, best_epoch, patience_cnt = em, epoch, 0
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"best_ep{epoch}_em{em:.4f}")
        model.save_pretrained(ckpt_path)
        tokenizer.save_pretrained(ckpt_path)
        for d in os.listdir(CHECKPOINT_DIR):
            full = os.path.join(CHECKPOINT_DIR, d)
            if full != ckpt_path and os.path.isdir(full):
                shutil.rmtree(full, ignore_errors=True)
        print(f"✅ Best EM mới → {best_em:.4f} (epoch {best_epoch})")
    else:
        patience_cnt += 1
        print(f"⚠️  Không cải thiện. (Đã TẮT Early Stopping - Sẽ tiếp tục chạy)")

    save_resume_checkpoint(
        model, optimizer, scheduler, scaler, epoch,
        global_step, best_em, best_epoch, patience_cnt, history
    )

    # ĐÃ XÓA ĐOẠN CODE EARLY STOPPING Ở ĐÂY ĐỂ MODEL CHẠY HẾT EPOCH

# ── 10. LƯU MODEL CUỐI ───────────────────────────────────────
print("\n>>> [6/7] Lưu model cuối...")
ckpt_dirs = sorted(
    [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith("best_")],
    key=lambda x: float(x.split("em")[-1])
)
if ckpt_dirs:
    best_dir = os.path.join(CHECKPOINT_DIR, ckpt_dirs[-1])
    AutoModelForSeq2SeqLM.from_pretrained(best_dir).save_pretrained(FINAL_SAVE_PATH)
    tokenizer.save_pretrained(FINAL_SAVE_PATH)

with open(os.path.join(FINAL_SAVE_PATH, "training_history.json"), "w") as f:
    json.dump({
        "config":     CFG,
        "history":    history,
        "best_em":    best_em,
        "best_epoch": best_epoch
    }, f, indent=2)

# Dọn dẹp resume
for path in [RESUME_STATE_FILE, RESUME_OPT_FILE]:
    if os.path.exists(path):
        os.remove(path)
shutil.rmtree(RESUME_MODEL_DIR, ignore_errors=True)
print("🗑️ Dọn dẹp checkpoint phụ thành công.")

# ── 11. INFERENCE + SPIDER OFFICIAL EVAL + PERFORMANCE METRICS ─
print("\n>>> [7/7] Inference & Spider Official Evaluation & Performance Metrics...")

# Load đúng model đã fine-tune từ FINAL_SAVE_PATH
infer_model = AutoModelForSeq2SeqLM.from_pretrained(FINAL_SAVE_PATH).to(DEVICE)
infer_model.eval()

# ── 11a. Batch Inference → pred.txt / gold.txt ────────────────
predictions, gold_lines = [], []

for batch in tqdm(dev_loader, desc="Inference", leave=False):
    with torch.no_grad():
        generated = infer_model.generate(
            input_ids=batch['input_ids'].to(DEVICE),
            attention_mask=batch['attention_mask'].to(DEVICE),
            max_new_tokens=CFG['max_target_len'],
            num_beams=CFG['beam_size'],
            early_stopping=True,
            length_penalty=CFG['length_penalty'],
        )
    preds = tokenizer.batch_decode(generated, skip_special_tokens=True)
    for pred, gold_sql, db_id in zip(preds, batch['target_sql'], batch['db_id']):
        predictions.append(pred + "\n")
        gold_lines.append(f"{gold_sql}\t{db_id}\n")

with open('pred.txt', 'w', encoding='utf-8') as f:
    f.writelines(predictions)
with open('gold.txt', 'w', encoding='utf-8') as f:
    f.writelines(gold_lines)

# ── 11b. Spider Official Evaluation ───────────────────────────
with open("evaluation.py", "r", encoding="utf-8") as f:
    eval_content = f.read()
eval_content = eval_content.replace(
    'conn = sqlite3.connect(db)',
    'conn = sqlite3.connect(db)\n    conn.text_factory = lambda b: b.decode(errors="ignore")'
)
with open("evaluation.py", "w", encoding="utf-8") as f:
    f.write(eval_content)

print("\n>>> Spider Official Results:")
os.system(
    "python evaluation.py "
    "--gold gold.txt --pred pred.txt "
    "--db spider_data/database "
    "--table spider_data/tables.json "
    "--etype all"
)

# ── 11c. Performance Metrics ──────────────────────────────────
print("\n>>> Đo Performance Metrics...")

# 1. Model Size (MB) — kích thước file weights trên disk
model_size_mb = measure_model_size_mb(FINAL_SAVE_PATH)

# 2. Latency (ms/sample) + Throughput (samples/s)
latency_ms, throughput_sps = measure_latency_throughput(
    infer_model, dev_loader, n_warmup_batches=3
)

# 3. Peak VRAM (MB) — đo trên 5 batch, reset counter trước khi đo
peak_vram_mb = measure_peak_vram_mb(infer_model, dev_loader, n_batches=5)

perf_metrics = {
    "model_size_mb":  round(model_size_mb, 2),
    "latency_ms":     latency_ms,
    "throughput_sps": throughput_sps,
    "peak_vram_mb":   peak_vram_mb,
}

# In bảng báo cáo
print_performance_report(perf_metrics)

# Lưu kèm vào training_history.json
history_path = os.path.join(FINAL_SAVE_PATH, "training_history.json")
with open(history_path, "r", encoding="utf-8") as f:
    saved_history = json.load(f)

saved_history["performance_metrics"] = perf_metrics
saved_history["best_em"]    = best_em
saved_history["best_epoch"] = best_epoch

with open(history_path, "w", encoding="utf-8") as f:
    json.dump(saved_history, f, indent=2)

# ── 11d. Lưu kết quả inference ────────────────────────────────
import shutil as _shutil
_shutil.copy('pred.txt', os.path.join(FINAL_SAVE_PATH, 'pred_dev.txt'))
_shutil.copy('gold.txt', os.path.join(FINAL_SAVE_PATH, 'gold_dev.txt'))

print(f"\n✅ Kết quả lưu thành công tại : {FINAL_SAVE_PATH}")
print(f"🏆 Best Exact Match           : {best_em:.4f} tại epoch {best_epoch}/{CFG['num_epochs']}")
print(f"💾 Model Size                 : {perf_metrics['model_size_mb']:.2f} MB")
print(f"⚡ Latency                    : {perf_metrics['latency_ms']:.2f} ms/sample")
print(f"🚀 Throughput                 : {perf_metrics['throughput_sps']:.2f} samples/s")
vram_display = (
    f"{perf_metrics['peak_vram_mb']:.2f} MB"
    if perf_metrics['peak_vram_mb'] > 0 else "N/A (CPU)"
)
print(f"🖥️  Peak VRAM                  : {vram_display}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 30.4 MB/s eta 0:00:00
>>> [1/7] Tải dữ liệu Spider...
Dataset URL: https://www.kaggle.com/datasets/jeromeblanchet/yale-universitys-spider-10-nlp-dataset
License(s): unknown
100% 96.0M/96.0M [00:00<00:00, 146MB/s]

Đang giải nén dữ liệu...
Đang tải công cụ chấm điểm chính thức của Spider...
Xong bước tải dữ liệu.
Thiết bị: cuda  |  Dùng FP16: True
>>> [2/7] Tiền xử lý dữ liệu...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Inference:   0%|          | 0/65 [00:00<?, ?it/s]


>>> Spider Official Results:

>>> Đo Performance Metrics...


⏱ Đo Latency/Throughput:   0%|          | 0/65 [00:00<?, ?it/s]

Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ce1db4b4b80>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
    self._shutdown_workers()
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
    if w.is_alive():
       ^^^^^^^^^^^^
  File "/usr/lib/python3.12/multiprocessing/process.py", line 160, in is_alive
    assert self._parent_pid == os.getpid(), 'can only test a child process'
 Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7ce1db4b4b80> 
Traceback (most recent call last):
   File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1707, in __del__
     self._shutdown_workers() 
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py", line 1690, in _shutdown_workers
          ^^^^if w.is_alive():^
^   ^  ^^ ^ ^^^^^^^^^^^^^^^


📊 PERFORMANCE METRICS REPORT
Chỉ số                               Giá trị
-------------------------------------------------------
Model Size (MB)                      230.77 MB
Latency (ms/sample)                  101.08 ms
Throughput (samples/s)                 9.89 s/s
Peak VRAM (MB)                    1808.70 MB


✅ Kết quả lưu thành công tại : /content/CodeT5-small
🏆 Best Exact Match           : 0.3046 tại epoch 5/5
💾 Model Size                 : 230.77 MB
⚡ Latency                    : 101.08 ms/sample
🚀 Throughput                 : 9.89 samples/s
🖥️  Peak VRAM                  : 1808.70 MB


In [5]:
import os

print("⏳ Đang chạy chấm điểm Spider và lưu kết quả...")
# 1. Chạy lệnh chấm điểm và chuyển hướng output (>) vào file eval_results.txt
colab_cmd = "python evaluation.py --gold gold.txt --pred pred.txt --db spider_data/database --table spider_data/tables.json --etype all > eval_results.txt"
os.system(colab_cmd)

# 2. Đọc file kết quả vừa lưu
with open("eval_results.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print("✅ Đã chấm xong! Dưới đây là trích xuất kết quả:\n")

# 3. IN RA CÁC CÂU LỖI (Mẫu khoảng 20 dòng đầu để tránh Colab bị đơ)
print("="*60)
print("🔍 TRÍCH XUẤT MỘT SỐ CÂU DỰ ĐOÁN SAI (PRED vs GOLD)")
print("="*60)

error_lines_printed = 0
for line in lines:
    if "pred:" in line or "gold:" in line or "eval_err_num" in line:
        print(line.strip())
        error_lines_printed += 1
        if error_lines_printed >= 20:  # Chỉ in 20 dòng đầu
            print("\n... (Còn nhiều câu sai khác, bạn có thể tải file 'eval_results.txt' về để xem hết) ...\n")
            break

# 4. IN RA BẢNG ĐIỂM TỔNG KẾT (METRICS)
print("="*60)
print("📊 BẢNG ĐIỂM CHI TIẾT (METRICS)")
print("="*60)

# Lấy 50 dòng cuối cùng trong file log (Đây là nơi chứa các bảng điểm order, and/or, keywords...)
for line in lines[-50:]:
    print(line.rstrip())

print("\n📁 Toàn bộ log chi tiết đã được lưu tại: /content/eval_results.txt")

⏳ Đang chạy chấm điểm Spider và lưu kết quả...
✅ Đã chấm xong! Dưới đây là trích xuất kết quả:

🔍 TRÍCH XUẤT MỘT SỐ CÂU DỰ ĐOÁN SAI (PRED vs GOLD)
medium pred: SELECT name ,  country ,  age FROM singer ORDER BY age
medium gold: SELECT name ,  country ,  age FROM singer ORDER BY age DESC
medium pred: SELECT name ,  capacity FROM stadium ORDER BY avg(Average) DESC LIMIT 1
medium gold: SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1
medium pred: SELECT name ,  capacity FROM stadium ORDER BY avg(Average) DESC LIMIT 1
medium gold: SELECT name ,  capacity FROM stadium ORDER BY average DESC LIMIT 1
extra pred: SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year  >  2013 GROUP BY T1.stadium_id ORDER BY count(*) DESC LIMIT 1
extra gold: SELECT T2.name ,  T2.capacity FROM concert AS T1 JOIN stadium AS T2 ON T1.stadium_id  =  T2.stadium_id WHERE T1.year  >=  2014 GROUP BY T2.stadium_id ORDER BY count(*) DESC LIMIT